# Bank Deposit Prediction
Decision Tree classifier with GridSearchCV hyperparameter tuning, precision-recall analysis,
and Random Forest comparison on Portuguese bank marketing dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, roc_auc_score
)
from scipy.stats import chi2_contingency

os.makedirs('images', exist_ok=True)

## 1. Load Data

In [ ]:
# Place bank.csv in same folder before running
data = pd.read_csv('bank.csv')
print('Shape:', data.shape)
data.head()

## 2. Data Quality Checks

In [ ]:
print('Null values:\n', data.isna().sum())
print('\nDuplicates:', data.duplicated().sum())
data.info()

## 3. Exploratory Data Analysis

In [ ]:
num_cols = [col for col in data.columns if data[col].dtype == 'int64']
cat_cols = [col for col in data.columns if col not in num_cols]

print('Numerical columns:', num_cols)
print('Categorical columns:', cat_cols)
data[num_cols].describe()

In [ ]:
# Boxplots for outlier detection
num_rows = (len(num_cols) + 2) // 3
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5*num_rows))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x=data[col], ax=axes[i])
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
fig.tight_layout()
plt.savefig('images/outlier_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots with deposit hue — outliers are meaningful
num_rows = (len(num_cols) + 2) // 3
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5*num_rows))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x=data[col], ax=axes[i], hue=data['deposit'])
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
fig.tight_layout()
plt.savefig('images/boxplots_by_deposit.png', bbox_inches='tight')
plt.show()
# Note: keeping outliers — Decision Tree is not sensitive to them

In [ ]:
# Categorical feature distributions
num_rows = (len(cat_cols) + 2) // 3
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5*num_rows))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    sns.countplot(x=data[col], ax=axes[i])
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45)
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
fig.tight_layout()
plt.savefig('images/categorical_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features vs deposit target
num_rows = (len(cat_cols[:-1]) + 2) // 3
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5*num_rows))
axes = axes.flatten()
for i, col in enumerate(cat_cols[:-1]):
    sns.countplot(x=data[col], ax=axes[i], hue=data['deposit'])
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=60)
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
fig.tight_layout()
plt.savefig('images/categorical_vs_deposit.png', bbox_inches='tight')
plt.show()

## 4. Feature Selection — Chi-Square Test

In [ ]:
# Chi-square test for categorical feature importance
def chi_square_test(data, cat_cols, target_col):
    results = []
    for cat_col in cat_cols:
        contingency_table = pd.crosstab(data[cat_col], data[target_col])
        chi2, p, dof, expected = chi2_contingency(contingency_table)
        results.append((cat_col, round(p, 6)))
    return sorted(results, key=lambda x: x[1])

chi_results = chi_square_test(data, cat_cols[:-1], 'deposit')
print('Chi-Square Test Results (p-value):')
for col, p in chi_results:
    print(f'  {col}: p={p} {"✓ significant" if p < 0.05 else "✗ not significant"}')
# All p < 0.05 → keep all categorical features

## 5. Preprocessing

In [ ]:
# Encode target
data['deposit'] = data['deposit'].map({'yes': 1, 'no': 0})
print('Target distribution:')
print(data['deposit'].value_counts())
# Fairly balanced — no need for oversampling

In [ ]:
# One-hot encoding for categorical features
X = data.drop(columns='deposit')
y = data['deposit']
X = pd.get_dummies(X, dtype='int')
# Note: drop_first=False intentional — tree models not affected by multicollinearity

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=43
)
print('Train shape:', X_train.shape)
print('Test shape: ', X_test.shape)

## 6. Baseline Decision Tree

In [ ]:
# Baseline — no tuning
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

train_score = model.score(X_train, y_train)
test_score  = model.score(X_test, y_test)
print(f'Train accuracy: {train_score:.4f}')
print(f'Test accuracy:  {test_score:.4f}')
print(f'Model depth:    {model.get_depth()}')
# Overfitting expected — train=1.0, test much lower

## 7. Hyperparameter Tuning — GridSearchCV

In [ ]:
# GridSearchCV to fix overfitting
params = {
    'max_depth':        list(range(4, 15)),
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':     ['sqrt', 'log2', None],  # fixed: removed invalid slice
    'criterion':        ['gini', 'entropy']
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)
print('Best parameters:', grid.best_params_)
print('Best CV score:  ', round(grid.best_score_, 4))

In [ ]:
# Train best Decision Tree
best_dt = grid.best_estimator_
y_pred       = best_dt.predict(X_test)
y_pred_train = best_dt.predict(X_train)

print(f'Train accuracy: {best_dt.score(X_train, y_train):.4f}')
print(f'Test accuracy:  {best_dt.score(X_test, y_test):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_train, y_pred_train), annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix - Train')
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix - Test')
plt.tight_layout()
plt.savefig('images/dt_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 8. Precision-Recall Curve — Threshold Analysis

In [ ]:
# Precision-recall tradeoff at different thresholds
y_pred_proba = best_dt.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precision[1:], label='Precision')
plt.plot(thresholds, recall[1:], label='Recall')
plt.axvline(x=0.2,  color='red',    linestyle='--', label='Threshold 0.2 (high recall)')
plt.axvline(x=0.5,  color='orange', linestyle='--', label='Threshold 0.5 (balanced)')
plt.axvline(x=0.95, color='green',  linestyle='--', label='Threshold 0.95 (high precision)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig('images/precision_recall_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# Compare three threshold strategies
for threshold, label in [(0.2, 'High Recall'), (0.5, 'Balanced F1'), (0.95, 'High Precision')]:
    y_new = [1 if p > threshold else 0 for p in y_pred_proba]
    print(f'\n--- Threshold {threshold} ({label}) ---')
    print(classification_report(y_test, y_new))
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_test, y_new), annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix — Threshold {threshold} ({label})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig(f'images/cm_threshold_{str(threshold).replace(".","_")}.png', bbox_inches='tight')
    plt.show()

## 9. Random Forest Comparison

In [ ]:
# Random Forest — compare with Decision Tree
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
print('Random Forest Results:')
print(f'  Train accuracy: {rf.score(X_train, y_train):.4f}')
print(f'  Test accuracy:  {rf.score(X_test, y_test):.4f}')
print(f'  ROC-AUC:        {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

print('\nDecision Tree (tuned) Results:')
print(f'  Train accuracy: {best_dt.score(X_train, y_train):.4f}')
print(f'  Test accuracy:  {best_dt.score(X_test, y_test):.4f}')
print(f'  ROC-AUC:        {roc_auc_score(y_test, y_pred_proba):.4f}')

In [ ]:
# Feature importance from Random Forest
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feat_imp)
plt.title('Top 15 Important Features — Random Forest')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.savefig('images/feature_importance.png', bbox_inches='tight')
plt.show()

## 10. Key Findings

1. Senior citizens show more tendency towards making a deposit
2. Longer call duration = higher chance of deposit subscription
3. Cellular contact is the most effective communication method
4. Clients with successful previous campaign outcome are more likely to subscribe
5. Decision Tree overfits without tuning — GridSearchCV reduces depth and improves generalization
6. Random Forest outperforms single Decision Tree as expected
7. Threshold 0.2 maximizes recall — better for acquisition campaigns (don't miss potential depositors)
8. Threshold 0.95 maximizes precision — better for targeted campaigns with limited resources